# Smart Budget — SageMaker Endpoint

**Ticket:** DATA-1140 · **Endpoint:** `smart-budget-suggestion-endpoint`

> El entorno de Studio tiene `sagemaker` SDK roto (`sagemaker_core` no importable).
> Este notebook usa **boto3 puro** — equivalente a `SKLearnModel.deploy()` pero sin el SDK.

| Step | Qué hace |
|------|---------|
| 1 | Empaquetar `inference.py` + código + datos en `model.tar.gz` |
| 2 | Subir a S3 |
| 3 | Crear modelo + endpoint en SageMaker |
| 4 | Probar el endpoint |


In [ ]:
import boto3, json, os, shutil, tarfile, time
from pathlib import Path

# Sesión — Studio usa credenciales del entorno; local usa perfil blossom-dev
try:
    session = boto3.Session()
    identity = session.client('sts').get_caller_identity()
    print("✅ SageMaker Studio")
except Exception:
    session = boto3.Session(profile_name='blossom-dev')
    identity = session.client('sts').get_caller_identity()
    print("✅ Local (blossom-dev)")

ACCOUNT_ID = identity['Account']
REGION     = session.region_name or 'us-east-1'

# Rol de ejecución — derivado desde la identidad actual (reemplaza get_execution_role)
_arn = identity['Arn']
if ':assumed-role/' in _arn:
    _role_name = _arn.split(':assumed-role/')[1].split('/')[0]
    ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{_role_name}"
else:
    ROLE_ARN = _arn

# Clientes
s3      = session.client('s3')
sm      = session.client('sagemaker')
runtime = session.client('sagemaker-runtime')

# Constantes
ENDPOINT_NAME = 'smart-budget-suggestion-endpoint'
S3_BUCKET     = 'blossom-analytics-safe-dev-nv'
S3_KEY        = 'smart_budget/endpoint/v1/model.tar.gz'
S3_URI        = f's3://{S3_BUCKET}/{S3_KEY}'

# Imagen SKLearn 1.2-1 equivalente a framework_version="1.2-1" en SKLearnModel
_SKLEARN_IMAGES = {
    'us-east-1': '683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3',
    'us-east-2': '257758044811.dkr.ecr.us-east-2.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3',
    'us-west-2': '246618743249.dkr.ecr.us-west-2.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3',
}
CONTAINER_IMAGE = _SKLEARN_IMAGES.get(REGION, _SKLEARN_IMAGES['us-east-1'])

print(f"Account : {ACCOUNT_ID}")
print(f"Region  : {REGION}")
print(f"Role    : {ROLE_ARN}")
print(f"Image   : {CONTAINER_IMAGE}")


---
## Step 1 — Empaquetar `model.tar.gz`

In [ ]:
REPO_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())

SRC_INFERENCE    = REPO_ROOT / 'src' / 'api' / 'inference.py'
SRC_SMART_BUDGET = REPO_ROOT / 'src' / 'smart_budget'
DATA_DIR         = REPO_ROOT / 'data' / 'dough'
ARTIFACTS_DIR    = REPO_ROOT / 'notebooks' / 'model_artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)

staging = ARTIFACTS_DIR / 'staging'
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir()

shutil.copy(SRC_INFERENCE, staging / 'inference.py')
shutil.copytree(SRC_SMART_BUDGET, staging / 'smart_budget')

data_staging = staging / 'data'
data_staging.mkdir()
for name in ['smart_budget_synthetic.csv']:
    src = DATA_DIR / name
    if src.exists(): shutil.copy(src, data_staging / name)
    else: print(f"⚠️  No encontrado: {src}")
for name in ['test_internal.csv', 'test_external.csv']:
    src = DATA_DIR / 'test' / name
    if src.exists(): shutil.copy(src, data_staging / name)
    else: print(f"⚠️  No encontrado: {src}")

tarball_path = ARTIFACTS_DIR / 'model.tar.gz'
with tarfile.open(tarball_path, 'w:gz') as tar:
    for item in staging.rglob('*'):
        if item.is_file():
            tar.add(item, arcname=item.relative_to(staging))

print(f"✅ model.tar.gz listo  ({tarball_path.stat().st_size / 1024:.1f} KB)")


---
## Step 2 — Subir a S3

In [ ]:
s3.upload_file(str(tarball_path), S3_BUCKET, S3_KEY)
print(f"✅ Subido: {S3_URI}")


---
## Step 3 — Crear modelo y endpoint

In [ ]:
MODEL_NAME  = f'smart-budget-model-{int(time.time())}'
CONFIG_NAME = f'smart-budget-config-{int(time.time())}'

# Equivalente a SKLearnModel(...) + deploy()
sm.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=ROLE_ARN,
    PrimaryContainer={
        'Image': CONTAINER_IMAGE,
        'ModelDataUrl': S3_URI,
        'Environment': {'SAGEMAKER_PROGRAM': 'inference.py'},
    },
)

sm.create_endpoint_config(
    EndpointConfigName=CONFIG_NAME,
    ProductionVariants=[{
        'VariantName':            'AllTraffic',
        'ModelName':              MODEL_NAME,
        'InitialInstanceCount':   1,
        'InstanceType':           'ml.m5.large',
        'InitialVariantWeight':   1,
    }],
)

# Crear o actualizar si ya existe
existing = [e['EndpointName'] for e in sm.list_endpoints()['Endpoints']]
if ENDPOINT_NAME in existing:
    sm.update_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=CONFIG_NAME)
    print(f"🔄 Actualizando: {ENDPOINT_NAME}")
else:
    sm.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=CONFIG_NAME)
    print(f"🚀 Creando: {ENDPOINT_NAME}")

print("Esperando InService... (3–5 min)")
sm.get_waiter('endpoint_in_service').wait(
    EndpointName=ENDPOINT_NAME,
    WaiterConfig={'Delay': 15, 'MaxAttempts': 40},
)
print(f"✅ Endpoint listo: {ENDPOINT_NAME}")


---
## Step 4 — Probar el endpoint

In [ ]:
def invoke(payload):
    r = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType='application/json',
        Body=json.dumps(payload),
    )
    return json.loads(r['Body'].read())

# Happy path
result = invoke({'idaccount': 'EXT2', 'defaultcategory': 'Food & Dining', 'period_id': '2026-05'})
print(json.dumps(result, indent=2))


In [ ]:
import botocore

# Regla 1 — Cuenta no existe → error
try:
    invoke({'idaccount': 'CUENTA_INEXISTENTE', 'defaultcategory': 'Groceries', 'period_id': '2026-05'})
    print("❌ Regla 1 FALLÓ")
except (runtime.exceptions.ModelError, botocore.exceptions.ClientError):
    print("✅ Regla 1 — cuenta inexistente → error")

# Regla 2 — Categoría inválida → error
try:
    invoke({'idaccount': 'EXT2', 'defaultcategory': 'CategoriaFalsa', 'period_id': '2026-05'})
    print("❌ Regla 2 FALLÓ")
except (runtime.exceptions.ModelError, botocore.exceptions.ClientError):
    print("✅ Regla 2 — categoría inválida → error")

# Regla 3 — Sin datos → null
r = invoke({'idaccount': 'SYN001', 'defaultcategory': 'Groceries', 'period_id': '2026-05'})
assert r['suggested_amount'] is None
print(f"✅ Regla 3 — sin datos → null  ({r.get('display_label')})")


---
## ⚠️ Borrar endpoint cuando termines

Genera costo por hora mientras está activo.

In [ ]:
sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
print(f"✅ Eliminado: {ENDPOINT_NAME}")
